In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
import gc
p_path = "/content/drive/My Drive/Campbell A data/preprocess_data.parquet"
LOCAL_PATH = '/content/df_processed.parquet'

# preprocess and saved to disk
if not os.path.exists(LOCAL_PATH):
    df = pd.read_parquet(p_path)

    # preprocess
    id_cols    = ['DATE', 'permno']
    float_cols = [c for c in df.columns if c not in id_cols]
    df[float_cols] = df[float_cols].astype(np.float32)

    # generate dummy variables
    sic_df        = pd.get_dummies(df['sic2'], prefix='sic')
    sic_cols_list = sic_df.columns.tolist()
    df            = pd.concat([df, sic_df], axis=1)

    # saved to disk
    df.to_parquet(LOCAL_PATH, index=False)
    print(f"finished preprocessing, saved to local disk")

# skip preprocess
else:
    df            = pd.read_parquet(LOCAL_PATH)
    sic_cols_list = [c for c in df.columns if c.startswith('sic_')]
    print(f"read from local disk, skip preprocessing")

print(f"df shape: {df.shape}")
def generate_920_features(df, char_cols, macro_cols, sic_cols):

    X_char = df[char_cols].to_numpy(dtype=np.float32, copy=False)
    X_macro = df[macro_cols].to_numpy(dtype=np.float32, copy=False)
    X_sic = df[sic_cols].to_numpy(dtype=np.float32, copy=False)

    # generate interaction (N x 752)

    X_inter = (X_char[:, :, np.newaxis] * X_macro[:, np.newaxis, :]).reshape(len(df), -1)

    # features (94) + interaction (752) + industry (74)
    X_920 = np.hstack([X_char, X_inter, X_sic])

    return X_920
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
features=list(df.columns)[2:96]

def calc_oos_r2(actual, predicted):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    denom = np.sum(actual ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((actual - predicted) ** 2) / denom

Mounted at /content/drive
Mounted at /content/drive
finished preprocessing, saved to local disk
df shape: (3712808, 183)
finished preprocessing, saved to local disk
df shape: (3712808, 183)


In [2]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

def select_n_components_pcr(X_train, y_train, X_val, y_val,
                            max_components=10, prev_best_n=None):

    max_components = min(max_components, X_train.shape[1], X_train.shape[0] - 1)

    if prev_best_n is None:
        search_range = range(1, max_components + 1)
    else:
        lo = max(1, prev_best_n - 1)
        hi = min(max_components, prev_best_n + 1)
        search_range = range(lo, hi + 1)

    best_r2, best_n = -np.inf, 1
    for n in search_range:
        # PCR is purely PCA dimension reduction followed by OLS
        pcr = Pipeline([
            ('pca', PCA(n_components=n)),
            ('reg', LinearRegression())
        ])
        pcr.fit(X_train, y_train)
        r2 = calc_oos_r2(y_val, pcr.predict(X_val).ravel())
        if r2 > best_r2:
            best_r2, best_n = r2, n

    return best_n

In [3]:
import warnings
import time
import gc
import pyarrow.parquet as pq
from sklearn.preprocessing import StandardScaler

start_test_year = 1987
end_test_year   = 2016
VAL_YEARS       = 12
MAX_COMPONENTS  = 10
all_preds   = []
prev_best_n = None
dates_all   = df['DATE'].values.copy()
y_all       = df['exret'].to_numpy(dtype=np.float32, copy=False)

del df
gc.collect()

needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret', 'mvel1']
needed_cols = list(dict.fromkeys(needed_cols))

for year in range(start_test_year, end_test_year + 1):
    print(f"\n--- cope with {year} year ---")
    t0 = time.time()

    # split data
    train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                 (dates_all <= pd.Timestamp(year - 13, 12, 31))
    val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year - 1, 12, 31))
    test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year, 12, 31))

    df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
    df_train = df_year[train_mask].reset_index(drop=True)
    df_val   = df_year[val_mask].reset_index(drop=True)
    df_test  = df_year[test_mask].reset_index(drop=True)
    del df_year
    gc.collect()

    X_train = generate_920_features(df_train, features, macro, sic_cols_list)
    y_train = y_all[train_mask]
    X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
    y_val   = y_all[val_mask]
    X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)
    del df_train, df_val
    gc.collect()

    t1 = time.time()
    print(f'  feature construction used: {t1-t0:.1f}s  '
          f'| train rows: {len(X_train):,}  val rows: {len(X_val):,}')

    # parameter selection
    scaler_sel = StandardScaler()
    X_train_s  = scaler_sel.fit_transform(X_train)
    X_val_s    = scaler_sel.transform(X_val)

    best_n = select_n_components_pcr(
        X_train_s, y_train, X_val_s, y_val,
        max_components=MAX_COMPONENTS,
        prev_best_n=prev_best_n,
    )
    del X_train_s, X_val_s, scaler_sel
    gc.collect()
    print(f"  best_n={best_n}  select parameter used: {time.time()-t1:.1f}s")

    # train
    X_trainval = np.vstack([X_train, X_val])
    y_trainval = np.concatenate([y_train, y_val])
    del X_train, X_val, y_train, y_val
    gc.collect()

    scaler_final = StandardScaler()
    X_trainval_s = scaler_final.fit_transform(X_trainval)
    X_test_s     = scaler_final.transform(X_test)
    del X_trainval, X_test
    gc.collect()

    t2 = time.time()

    # Using the PCR Pipeline
    final_model = Pipeline([
        ('pca', PCA(n_components=best_n)),
        ('reg', LinearRegression())
    ])

    final_model.fit(X_trainval_s, y_trainval)
    del X_trainval_s, y_trainval
    gc.collect()
    print(f"  train used: {time.time()-t2:.1f}s  |  total: {time.time()-t0:.1f}s")

    # predict
    res = df_test[['DATE', 'permno', 'exret', 'mvel1']].copy().reset_index(drop=True)
    res['y_pred'] = final_model.predict(X_test_s).ravel()
    del X_test_s, final_model, df_test, scaler_final
    gc.collect()

    all_preds.append(res)
    prev_best_n = best_n

# Re-run your final evaluation report exactly as you had it below!


--- cope with 1987 year ---
  feature construction used: 5.4s  | train rows: 472,278  val rows: 764,497
  best_n=1  select parameter used: 33.3s
  train used: 3.8s  |  total: 58.1s

--- cope with 1988 year ---
  feature construction used: 5.7s  | train rows: 530,435  val rows: 788,744
  best_n=2  select parameter used: 14.8s
  train used: 4.5s  |  total: 41.5s

--- cope with 1989 year ---
  feature construction used: 5.9s  | train rows: 588,534  val rows: 814,060
  best_n=2  select parameter used: 17.1s
  train used: 4.6s  |  total: 45.2s

--- cope with 1990 year ---
  feature construction used: 6.1s  | train rows: 647,363  val rows: 836,447
  best_n=2  select parameter used: 20.3s
  train used: 4.9s  |  total: 49.8s

--- cope with 1991 year ---
  feature construction used: 6.4s  | train rows: 704,916  val rows: 859,101
  best_n=1  select parameter used: 21.4s
  train used: 4.8s  |  total: 52.2s

--- cope with 1992 year ---
  feature construction used: 6.7s  | train rows: 761,970  val

In [4]:
results = pd.concat(all_preds, ignore_index=True)

r2_all = calc_oos_r2(results['exret'], results['y_pred'])

top1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, False])
    .groupby('DATE', sort=False).head(1000)
)
r2_top = calc_oos_r2(top1000['exret'], top1000['y_pred'])

bot1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, True])
    .groupby('DATE', sort=False).head(1000)
)
r2_bot = calc_oos_r2(bot1000['exret'], bot1000['y_pred'])

print(f"\n{'='*45}")
print(f"  {'Subsample':<25}  {'OOS R2':>10}")
print(f"{'-'*45}")
print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
print(f"{'='*45}")

export_path = '/content/drive/MyDrive/pcr_predictions.parquet'
results.to_parquet(export_path, index=False)
print(f"Saved PCR predictions to Google Drive: {export_path}")


  Subsample                      OOS R2
---------------------------------------------
  All stocks                    +0.1639%
  Top 1000 (largest)            +0.3512%
  Bottom 1000 (smallest)        +0.2673%
Saved PCR predictions to Google Drive: /content/drive/MyDrive/pcr_predictions.parquet
